# Driver — độ bền của static Android malware detection dưới obfuscation

Notebook này chỉ **điều khiển**; mọi logic nằm trong `src/`. Chạy tuần tự theo phase.

| Phase | Cell | Thời gian | Chạy lại mỗi session? |
|---|---|---|---|
| 0 | Setup + smoke test | 30 phút | Có |
| 1 | Tải dataset + manifest + split | ~30 phút | Có (trừ manifest/split đã ở Drive) |
| 2 | Trích feature APK sạch | ~1,1 giờ | Không — checkpoint ở Drive |
| 3 | Baseline ML + eval sạch | ~1 giờ | Không |
| 4 | Obfuscation | 9–10 giờ, **chia 2 session** | Resume từ `obf_progress.json` |
| 5 | Ma trận kết quả | ~2 giờ | Không |

**Đừng đi tiếp nếu smoke test ở Phase 0 hỏng.**

## Phase 0 — Setup

In [ ]:
# Mount Drive để checkpoint
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/apk-robustness'
SCRATCH = '/content/apkrob'

import os
os.environ['APKROB_WORK'] = WORK
os.environ['APKROB_SCRATCH'] = SCRATCH
os.makedirs(WORK, exist_ok=True)
print('WORK   ', WORK)
print('SCRATCH', SCRATCH)

In [ ]:
# Java + Android build-tools cho Obfuscapk.
# apktool KHÔNG có trong danh sách gốc của PLAN nhưng Obfuscapk bắt buộc phải có —
# thiếu nó thì mọi kỹ thuật đều fail ngay ở bước decompile.
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless apktool zipalign apksigner

### Obfuscapk — `pip install obfuscapk` không bao giờ chạy được

PLAN mục 2 ghi `pip install obfuscapk`, nhưng lệnh đó luôn thất bại: Obfuscapk **không phát hành trên PyPI**, và repo cũng không có `setup.py` ở gốc nên `pip install git+...` cũng không được. Cách duy nhất là clone rồi đưa thư mục `src/` của nó vào `PYTHONPATH`.

Còn một cái bẫy thứ hai nằm ngay sau đó: `src/requirements.txt` của Obfuscapk ghim `Yapsy==1.12.2` — bản mới nhất trên PyPI, phát hành 2019 — và bản đó `import imp`, module đã **bị xoá khỏi Python 3.12**. Colab đang chạy đúng phiên bản này. Nhánh master của Yapsy đã chuyển sang `importlib` nhưng chưa bao giờ được phát hành, nên phải cài Yapsy từ git.

Các pin còn lại của Obfuscapk cũng từ 2021 (`pycryptodome==3.12.0`) và không có wheel cho Python 3.12, nên cell dưới cài bản mới nhất thay vì theo pin.

Cell "Kiểm tra toolchain" ở dưới sẽ báo chính xác cái nào trong hai cái này hỏng, nếu có.

In [ ]:
import os, sys
print('Python', sys.version.split()[0], '- Yapsy tren PyPI hong tu 3.12 tro len')

# 1. Obfuscapk: clone, KHÔNG pip install (không có trên PyPI)
OBF = '/content/Obfuscapk'
if not os.path.isdir(OBF):
    !git clone -q --depth 1 https://github.com/ClaudiuGeorgiu/Obfuscapk.git {OBF}
os.environ['OBFUSCAPK_SRC'] = f'{OBF}/src'   # src/obfuscate.py đọc biến này

# 2. Phụ thuộc của Obfuscapk — bỏ pin cũ, trừ Yapsy phải lấy từ git
!pip -q install pycryptodome tqdm vt-py
!pip -q install 'yapsy @ git+https://github.com/tibonihoo/yapsy.git@master#subdirectory=package'

# 3. Phụ thuộc của dự án này
!pip -q install androguard==4.1.2 scikit-learn pandas pyarrow xgboost networkx joblib

print('OBFUSCAPK_SRC =', os.environ['OBFUSCAPK_SRC'])

In [ ]:
# Lấy code của dự án này.
REPO = 'https://github.com/trantrien1/AndroiDetection1.git'
PROJ = '/content/AndroiDetection1'

import os
if not os.path.isdir(PROJ):
    !git clone -q {REPO} {PROJ}
else:
    !git -C {PROJ} pull -q          # lấy bản mới nhất khi chạy lại session

%cd {PROJ}
if not os.path.isdir('src'):
    raise SystemExit(f'Chưa có code trong {PROJ}/src — kiểm tra lại REPO.')
print('OK:', sorted(os.listdir('src')))

In [ ]:
# Kiểm tra toolchain TRƯỚC khi tải dataset — apktool, apksigner, zipalign và
# bản thân Obfuscapk. Cell này báo chính xác cái nào hỏng và cách sửa.
!python -c "
from src.obfuscate import check_toolchain
t = check_toolchain()
bad = [k for k, v in t.items() if not v]
print()
print('THIEU / HONG:', bad if bad else 'khong co - san sang')
raise SystemExit(1 if bad else 0)
"

## Phase 1 — Tải dataset + manifest + chốt split

In [ ]:
!python -m src.download --list        # xem server có gì trước khi tải 17k APK
!python -m src.download

In [ ]:
# Chốt split. Sau lần chạy đầu, test_sha256.txt KHÔNG BAO GIỜ đổi nữa:
# các lần chạy sau sẽ tự đọc lại file đã chốt.
!python -m src.split

In [ ]:
# SMOKE TEST — PLAN mục 2: chạy Obfuscapk trên 1 APK với Rebuild.
# Nếu cell này fail thì DỪNG LẠI, mọi thứ sau đều vô nghĩa.
!python -m src.obfuscate --smoke-test

## Phase 2 — Trích feature tĩnh

Chỉ trích 7.500 APK trong split đã chốt, không phải cả 17k. Checkpoint mỗi 500 APK xuống Drive nên cell này resume được sau khi session chết.

In [ ]:
# Thử 20 APK trước để biết tốc độ thực tế và tỉ lệ fail
!python -m src.features.extract --tag clean --limit 20

In [ ]:
!python -m src.features.extract --tag clean
!cat $APKROB_WORK/features/extract_stats_clean.json

## Phase 3 — Baseline ML + eval sạch

In [ ]:
# Train model toàn bộ feature (rf/xgb/svm) + model chỉ-một-nhóm cho Bảng B
!python -m src.train

In [ ]:
!python -m src.evaluate --val --clean

In [ ]:
# Baseline đối chiếu trên CSV gốc của CIC — con số này so được trực tiếp với
# literature. Tải CSV từ trang CICMalDroid rồi trỏ đường dẫn vào đây.
CSV = '/content/drive/MyDrive/apk-robustness/feature_vectors_syscallsbinders_frequency_5_Cat.csv'
!test -f "$CSV" && python -m src.train --csv-baseline "$CSV" || echo 'Chưa có CSV — bỏ qua bước đối chiếu'

## Phase 4 — Obfuscation (9–10 giờ, chia 2 session)

Session A chạy T1–T3, session B chạy T4–T6. `obf_progress.json` nằm trên Drive nên chạy lại cell là resume, không làm lại từ đầu.

In [ ]:
# SESSION A
!python -m src.obfuscate --techniques T1_trivial T2_rename T3_string --workers 4

In [ ]:
# SESSION B
!python -m src.obfuscate --techniques T4_asset T5_cfg T6_reflection --workers 4

In [ ]:
!python -m src.obfuscate --report   # tỉ lệ APK hỏng theo từng kỹ thuật

In [ ]:
# Trích feature trên APK đã obfuscate.
# APK obfuscated nằm ở SCRATCH nên phải chạy cell này TRONG CÙNG session
# với cell obfuscate ở trên, trước khi Colab xoá disk.
for tech in ['T1_trivial','T2_rename','T3_string','T4_asset','T5_cfg','T6_reflection']:
    !python -m src.features.extract --tag {tech}

## Phase 5 — Ma trận kết quả

In [ ]:
!python -m src.evaluate --obf

In [ ]:
!python -m src.matrix

In [ ]:
from IPython.display import Markdown, display
import os
display(Markdown(open(os.path.join(os.environ['APKROB_WORK'], 'results', 'tables.md'), encoding='utf-8').read()))

## Pass sau — chỉ khi Bảng B đòi

`G5` (call graph) tốn 5–30s/APK, tức ~80% tổng thời gian Phase 2. Chỉ bật nếu Bảng B cho thấy G1–G4 không đủ tách bạch.

In [ ]:
# !python -m src.features.extract --tag clean --with-g5
# !python -m src.train --groups G1 G2 G3 G4 G5